# AdaRetrieve — controlled T4 test
Use the V5 AdaRetrieve-project.zip and AdaRetrieve-indexes-and-scores.zip. The notebook verifies both archives and never updates the source release. The reported routing latency charges each arm the full measured common prefix plus its own measured suffix.

In [ ]:
import os,sys, tempfile,zipfile,hashlib,json,importlib.util,subprocess
from pathlib import Path
from google.colab import files
import torch
assert torch.cuda.is_available() and 'T4' in torch.cuda.get_device_name(), 'Select a Tesla T4 and reconnect.'
content=Path('/content')
expected={'AdaRetrieve-project.zip':'3409513b06232df6c6bbab7bb11903bec85522af17f80fc039a09db833409b2c','AdaRetrieve-indexes-and-scores.zip':'3af2b67a10413a7b6e7de78aea782d1335cb999af380c8b0bb00f89a20176340'}
if not all((content/n).is_file() for n in expected):
    print('Upload both original ZIP files to the runtime.')
    files.upload()
for n,h in expected.items():
    assert hashlib.sha256((content/n).read_bytes()).hexdigest()==h,n
work=Path(tempfile.mkdtemp(prefix='controlled-',dir=content))
with zipfile.ZipFile(content/'AdaRetrieve-project.zip') as z:
    for n in z.namelist():
        if n.startswith(('project/','routing-followup/')): z.extract(n,work)
with zipfile.ZipFile(content/'AdaRetrieve-indexes-and-scores.zip') as z:
    for n in z.namelist():
        if n.endswith(('.bm.pkl','.faiss')): z.extract(n,work/'project/cache')
if importlib.util.find_spec('faiss') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','faiss-cpu'],check=True)
print('VERIFIED',torch.cuda.get_device_name(),work,flush=True)

In [ ]:
benchmark_source = '"""Paired T4 fork and fixed-RRF/reranker benchmark with frozen V5 folds."""\nimport hashlib\nimport json\nimport os\nimport random\nimport sys\nfrom collections import defaultdict\nfrom importlib.metadata import version\nfrom pathlib import Path\nfrom time import perf_counter_ns\n\nimport faiss\nimport torch\nfrom transformers import AutoModelForSequenceClassification, AutoTokenizer\n\nBASE = Path(__file__).resolve().parent\nsys.path.insert(0, str(BASE / "routing-followup"))\nfrom experiment import ReuseEngine, Revision, Router, State\nfrom neural import CE, ENC, Models\nfrom retrieval import Collection, DOMAINS, fused, metrics, order\nfrom study import Engine\n\nL6 = ("cross-encoder/ms-marco-MiniLM-L6-v2", "233902d25c440f23af6f7d6e94d2946bac0bee0a")\nOUTPUT = BASE / "controlled-output"\nFIXED = ("RRF", "Tiny8", "L6-8", "Tiny24", "L6-24")\nnow = perf_counter_ns\n\n\ndef elapsed(start):\n    return (now() - start) / 1e6\n\n\ndef timed(stages, name, fn, *args):\n    start = now()\n    result = fn(*args)\n    stages[name] += elapsed(start)\n    return result\n\n\ndef run_actions(state, actions, get, score, clock, budget, profile, stages, fork):\n    for i, action in enumerate(actions):\n        if action not in state.legal() or clock() + profile[action] > budget:\n            break\n        state = timed(stages, action, state.next, action, get, score)\n        if fork and action == "dense20":\n            return state, actions[i+1:], True\n    return state, (), False\n\n\ndef record(state, total, stages, **extra):\n    return dict(top=state.rank[:10], actions=list(state.path), ms=total,\n                stages=dict(stages), **extra)\n\n\ndef paired(reuse, text, budget, profile, root, revision, threshold, arms):\n    stages = defaultdict(float)\n    start = now()\n    get, score = reuse.operations(text)\n    state = timed(stages, "initial", lambda: State(text, get("initial")))\n    def select():\n        t = elapsed(start)\n        return tuple(root.paths[int(root.scores(state.features(budget, t), budget, t).argmax())])\n    plan = timed(stages, "root", select)\n    state, suffix, reached = run_actions(state, plan, get, score, lambda: elapsed(start),\n                                         budget, profile, stages, True)\n    shared_ms = elapsed(start)\n    info = dict(plan=plan, prefix=state.path, shared_ms=shared_ms,\n                remaining_ms=budget-shared_ms, reached_dense=reached)\n    if not reached:\n        return {arm:record(state, shared_ms, stages, changed=False, **info) for arm in arms}\n    output = {}\n    for arm in arms:\n        branch = stages.copy()\n        path, changed = suffix, False\n        begin = now()\n        if arm == "Revision" and threshold is not None:\n            path, changed = timed(branch, "revision_decision", revision.choose, state, budget,\n                                  shared_ms, suffix, profile, threshold)\n        end, _, _ = run_actions(state, path, get, score, lambda: shared_ms + elapsed(begin),\n                                budget, profile, branch, False)\n        total = shared_ms + elapsed(begin)\n        output[arm] = record(end, total, branch, changed=bool(changed), **info)\n    return output\n\n\ndef fixed(engine, scorers, text, label):\n    stages = defaultdict(float)\n    start = now()\n    get, _ = engine.operations(text)\n    bm = timed(stages, "bm25_64", get, "lex64")[0]\n    de = timed(stages, "dense64", get, "dense64")[0]\n    candidates = timed(stages, "rrf", fused, bm, de)\n    rank = candidates\n    if label != "RRF":\n        depth = 8 if label.endswith("8") else 24\n        ids = candidates[:depth]\n        score_model = scorers["Tiny" if label.startswith("Tiny") else "L6"]\n        def rerank():\n            logits = score_model([text]*len(ids), [engine.ds.text[i] for i in ids])\n            return [doc for _,doc in sorted(zip(logits,ids),key=lambda x:-x[0])] + candidates[depth:]\n        rank = timed(stages, "rerank", rerank)\n    total = elapsed(start)\n    return dict(ms=total, stages=dict(stages), top=rank[:10]), candidates\n\n\ndef read(path):\n    return json.loads(path.read_text())\n\n\ndef sha(path):\n    return hashlib.sha256(path.read_bytes()).hexdigest()\n\n\ndef main():\n    assert torch.cuda.is_available() and "T4" in torch.cuda.get_device_name()\n    faiss.omp_set_num_threads(1)\n    OUTPUT.mkdir(exist_ok=False)\n    gpu=torch.cuda.get_device_name()\n    print("HARDWARE", gpu, flush=True)\n    models=Models()\n    class L6Scorer:\n        score=Models.score\n    l6=L6Scorer()\n    l6.pairtoken=AutoTokenizer.from_pretrained(L6[0], revision=L6[1])\n    l6.ce=AutoModelForSequenceClassification.from_pretrained(L6[0],revision=L6[1]).half().cuda().eval()\n    scorers={"Tiny":models.score,"L6":l6.score}\n    project=BASE/"project"\n    follow=BASE/"routing-followup"/"output"\n    gate=read(follow/"source-gate.json")\n    env=dict(gpu=gpu,models=[ENC,CE,L6],code_sha256=sha(Path(__file__)),max_tokens=192,\n             versions={k:version(k) for k in ("numpy","torch","transformers","faiss-cpu")},\n             cpu_threads=torch.get_num_threads(),faiss_threads=faiss.omp_get_max_threads(),\n             session=os.urandom(12).hex(),clock="perf_counter_ns")\n    (OUTPUT/"environment.json").write_text(json.dumps(env,indent=2))\n    for d in DOMAINS:\n        foldpath=follow/f"frozen-{d}.json"\n        fold=read(foldpath)\n        rootpath=project/"results"/f"{d}.router.json"\n        assert sha(foldpath)==gate["frozen_hashes"][d] and sha(rootpath)==fold["root_sha256"]\n        assert set(fold["sources"])==set(DOMAINS)-{d} and fold["target"]==d\n        profile=fold["profile"]\n        spec=read(rootpath)\n        root=Router({**spec,"cost":[sum(profile[a] for a in path) for path in spec["paths"]]})\n        revision=Revision(fold["models"],profile)\n        budgets=fold["budgets"]\n        thresholds={r["budget"]:r["threshold"] for r in fold["calibration"]}\n        ds=Collection(project/"data/verified",d)\n        engine=Engine(ds,models)\n        reuse=ReuseEngine(engine)\n        labels=ds.labels("test")\n        warm=ds.source_ids()[0][:5]\n        assert not set(warm)&set(labels)\n        for q in warm:\n            for m in FIXED:\n                fixed(engine,scorers,ds.queries[q],m)\n            for b in budgets:\n                paired(reuse,ds.queries[q],b,profile,root,revision,thresholds[b],["OneShot","Revision"])\n        with (OUTPUT/f"{d}.jsonl").open("x") as writer:\n            for i,q in enumerate(order(labels,"controlled-v1"),1):\n                rng=random.Random(f"controlled-v1|{d}|{q}")\n                jobs=list(FIXED)+budgets\n                rng.shuffle(jobs)\n                results=[]; reference=None\n                for job in jobs:\n                    if isinstance(job,int):\n                        arms=["OneShot","Revision"]\n                        rng.shuffle(arms)\n                        pair=paired(reuse,ds.queries[q],job,profile,root,revision,thresholds[job],arms)\n                        results.extend(dict(method=m,budget=job,**r) for m,r in pair.items())\n                    else:\n                        r,candidates=fixed(engine,scorers,ds.queries[q],job)\n                        if reference is None: reference=candidates\n                        assert candidates==reference, (d,q,job)\n                        results.append(dict(method=job,budget=None,**r))\n                for r in results:\n                    r["ndcg"]=metrics(r["top"],ds,labels[q])["ndcg"]\n                writer.write(json.dumps(dict(q=q,rows=results),separators=(",",":"))+"\\n")\n                if i%100==0:\n                    writer.flush()\n                    print("PROGRESS",d,i,len(labels),flush=True)\n        print("FOLD_DONE",d,len(labels),flush=True)\n    print("ALL_FOLDS_DONE",flush=True)\n\n\nif __name__=="__main__":\n    main()\n'
(work/'run.py').write_text(benchmark_source,encoding='utf8')
subprocess.run([sys.executable,'-u',str(work/'run.py')],cwd=work,check=True)

In [ ]:
result=content/'AdaRetrieve-controlled-T4-results.zip'
with zipfile.ZipFile(result,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted((work/'controlled-output').glob('*')): z.write(p,p.name)
print('RESULTS_READY',result.stat().st_size,hashlib.sha256(result.read_bytes()).hexdigest(),flush=True)
files.download(str(result))